# BPClassifier — Boilerplate vs. Substantive Sentence Classifier

End-to-end pipeline for the BPClassifier course assignment.

**Pipeline steps**
1. Sentence extraction from earnings-call transcripts
2. Multi-judge gold labeling (Sonnet × 2 personas + Haiku) with disagreement audit
3. Stratified 60 / 20 / 20 split (frozen test)
4. Feature engineering: ~30 regex flags + frozen sentence embeddings
5. Classifier zoo: 12 entries across 7 families
6. 5-fold OOF threshold tuning under substantive recall ≥ 0.96
7. Held-out test evaluation and leaderboard
8. Save winning bundle for the GUI

**Alignment with the four focus items in the handout**

| Focus item | Where it lives in this notebook |
|---|---|
| Gold quality | §2 (three judges with distinct rubrics) and §3 (stratified disagreement audit) |
| Substantive recall ≥ 0.96 | §7 (OOF threshold sweep with hard floor; failures flagged, not relaxed) |
| Leaderboard breadth | §6 (12 entries, same features and splits, time + sent/sec reported) |
| GUI-ready model | §9 (joblib bundle: model + threshold + feature pipeline) |

**Re-runnable.** Every expensive step caches to Parquet (sentences, judge votes, embeddings) or joblib (models). Interrupt and re-run safely.


## 0 · Setup

Install dependencies if needed (uncomment the cell below). The pinned versions in `requirements.txt` are what this notebook was tested against.


In [2]:
# !pip install -q -r requirements.txt
# import nltk; nltk.download('punkt_tab', quiet=True); nltk.download('punkt', quiet=True)


In [3]:
from __future__ import annotations

import hashlib
import json
import os
import random
import re
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable

import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"sentence_transformers\.cross_encoder.*")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")

# --- Paths -----------------------------------------------------------------
PROJECT_ROOT = Path(".").resolve()
TRANSCRIPTS_DIR = PROJECT_ROOT / "transcripts"            # <-- DROP YOUR .txt FILES HERE
CACHE_DIR       = PROJECT_ROOT / "cache"
MODELS_DIR      = PROJECT_ROOT / "models"
REPORTS_DIR     = PROJECT_ROOT / "reports"
for d in (CACHE_DIR, MODELS_DIR, REPORTS_DIR):
    d.mkdir(exist_ok=True, parents=True)

# --- Reproducibility -------------------------------------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# --- Pipeline knobs --------------------------------------------------------
GOLD_POOL_SIZE         = 1500          # how many sentences to send to the LLM judges
MIN_SENT_CHARS         = 40            # drop fragments shorter than this
RECALL_FLOOR           = 0.96          # hard floor for substantive recall
EMBED_MODEL_NAME       = "sentence-transformers/all-mpnet-base-v2"
ENABLE_SETFIT          = True    # run SetFit strong-model pass
ENABLE_FINBERT         = True   # keep off for now; CPU-heavy overnight path
N_FOLDS                = 5
PARALLEL_JUDGE_WORKERS = 6             # be polite with the API

# --- Anthropic ------------------------------------------------------------
# Drop your key here OR set ANTHROPIC_API_KEY in your shell before launching Jupyter.
os.environ.setdefault("ANTHROPIC_API_KEY", "PASTE_YOUR_KEY_HERE")

JUDGE_MODELS = {
    "sonnet_balanced": "claude-sonnet-4-6",   # primary balanced reviewer
    "sonnet_skeptic":  "claude-sonnet-4-6",   # same model, stricter rubric persona
    "haiku_pattern":   "claude-haiku-4-5",    # fast pattern-focused reviewer
}

print("Project root:", PROJECT_ROOT)
print("Transcripts dir exists:", TRANSCRIPTS_DIR.exists(),
      f"({len(list(TRANSCRIPTS_DIR.glob('*.txt'))) if TRANSCRIPTS_DIR.exists() else 0} files)")


Project root: /Users/chaithanyapakala/Documents/NLP/Pakala_Chaithanya_NLP_HW2
Transcripts dir exists: True (131 files)


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 · Sentence extraction

Read every `.txt` transcript, split paragraphs, sentence-tokenize with NLTK punkt, dedupe, and drop short fragments. Saves to Parquet so we never repeat the work.


In [4]:
import nltk
try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)
    nltk.download("punkt", quiet=True)

from nltk.tokenize import sent_tokenize


def extract_sentences(transcripts_dir: Path, min_chars: int = MIN_SENT_CHARS) -> pd.DataFrame:
    rows = []
    for fp in sorted(transcripts_dir.glob("*.txt")):
        text = fp.read_text(encoding="utf-8", errors="ignore")
        # Paragraph split first (preserves speaker turns better than naive tokenization)
        for para in (p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()):
            for sent in sent_tokenize(para):
                sent = re.sub(r"\s+", " ", sent).strip()
                if len(sent) < min_chars:
                    continue
                rows.append({
                    "transcript": fp.stem,
                    "sentence": sent,
                    "char_len": len(sent),
                })

    df = pd.DataFrame(rows).drop_duplicates(subset=["sentence"]).reset_index(drop=True)
    df["sentence_id"] = df["sentence"].apply(
        lambda s: hashlib.md5(s.encode("utf-8")).hexdigest()[:12]
    )
    return df[["sentence_id", "transcript", "sentence", "char_len"]]


SENTENCES_PARQUET = CACHE_DIR / "sentences.parquet"

if SENTENCES_PARQUET.exists():
    sentences_df = pd.read_parquet(SENTENCES_PARQUET)
    print(f"Loaded {len(sentences_df):,} sentences from cache.")
else:
    assert TRANSCRIPTS_DIR.exists(), f"Place .txt transcripts in {TRANSCRIPTS_DIR}"
    sentences_df = extract_sentences(TRANSCRIPTS_DIR)
    sentences_df.to_parquet(SENTENCES_PARQUET, index=False)
    print(f"Extracted {len(sentences_df):,} sentences from "
          f"{sentences_df['transcript'].nunique()} transcripts.")

sentences_df.head(5)


Loaded 55,636 sentences from cache.


,sentence_id,transcript,sentence,char_len
0,4bbc8d818ab7,AMD_Q1-2024,"﻿Advanced Micro Devices, Inc., Q1 2024 Earning...",68
1,68f96bc4d200,AMD_Q1-2024,Presentation Operator Message Operator Greetin...,108
2,68635a3ee46e,AMD_Q1-2024,"[Operator Instructions] As a reminder, this co...",73
3,654a46e7cada,AMD_Q1-2024,"It is now my pleasure to introduce your host, ...",93
4,4b772535d61e,AMD_Q1-2024,Presenter Speech Executives - Former Vice Pres...,159


## 2 · Gold labeling — three judges, three rubrics

Per the handout, label quality is the single biggest credibility lever. We use **three distinct judges** (two prompt personas on Sonnet + one fast pass with Haiku) and majority-vote.

| Judge | Model | Persona |
|---|---|---|
| `sonnet_balanced` | claude-sonnet-4-6 | Balanced reviewer following the rubric strictly |
| `sonnet_skeptic`  | claude-sonnet-4-6 | Materiality skeptic — defaults to boilerplate when the sentence carries no specific information a downstream analyst would keep |
| `haiku_pattern`   | claude-haiku-4-5  | Fast pattern-detector — focuses on lexical cues |

Two distinct models × three rubric personas → genuinely distinct error modes. Each judge returns strict JSON with a label, a 0–1 confidence, and a one-line rationale. All outputs are cached per `(sentence_id, judge)` so you can interrupt and resume.

**Cost & resilience features built in:**
- **Prompt caching** on the rubric (the long, repeated part) — drops a 1,500-sentence run from ~$10 to ~$3.40.
- **Disk flush every 25 votes** — if the kernel dies you lose at most 25 calls.
- **Insufficient-balance detection** — if you run out of credits mid-run, the loop exits cleanly with a clear message instead of burning retries on every parallel worker. Top up, re-run the cell, it resumes.
- **`switch_api_key("sk-ant-...")`** helper — hot-swap the key without restarting the kernel. (Same account top-ups don't need this; it's only for genuinely new keys.)
- **Ctrl-C safe** — flushes pending votes before exiting.


In [5]:
# ---- Sample the gold pool ------------------------------------------------
GOLD_POOL_PARQUET = CACHE_DIR / "gold_pool.parquet"

if GOLD_POOL_PARQUET.exists():
    gold_pool = pd.read_parquet(GOLD_POOL_PARQUET)
    print(f"Loaded gold pool of {len(gold_pool):,} sentences from cache.")
else:
    # Stratify across transcripts so no single call dominates the gold set.
    rng = np.random.default_rng(SEED)
    per_transcript = max(1, GOLD_POOL_SIZE // max(1, sentences_df["transcript"].nunique()))
    sampled = (
        sentences_df.groupby("transcript", group_keys=False)
                    .apply(lambda g: g.sample(min(len(g), per_transcript), random_state=SEED))
    )
    if len(sampled) > GOLD_POOL_SIZE:
        sampled = sampled.sample(GOLD_POOL_SIZE, random_state=SEED)
    elif len(sampled) < GOLD_POOL_SIZE:
        # top up from the remainder
        remainder = sentences_df.drop(sampled.index)
        topup = remainder.sample(min(GOLD_POOL_SIZE - len(sampled), len(remainder)),
                                 random_state=SEED)
        sampled = pd.concat([sampled, topup])
    gold_pool = sampled.reset_index(drop=True)
    gold_pool.to_parquet(GOLD_POOL_PARQUET, index=False)
    print(f"Sampled {len(gold_pool):,} sentences for gold labeling.")

gold_pool.head(3)


Loaded gold pool of 1,500 sentences from cache.


,sentence_id,transcript,sentence,char_len
0,a5c9476fbedc,AMD_Q1-2024,"As I said, we have great customer engagements ...",69
1,cd7521ab9a17,AMD_Q1-2024,"Looking further ahead, AI represents an unprec...",74
2,e6a7bb29a86c,AMD_Q1-2024,"So overall, will help the mix on the gross mar...",55


In [6]:
# ---- Rubrics with anchor examples ---------------------------------------
RUBRIC_DEFINITIONS = """\
TASK
You are labeling a single sentence from an earnings-call transcript as one of two classes.

CLASSES
- "boilerplate": scripted intros, safe-harbor / forward-looking-statement language,
  operator and analyst housekeeping, generic thanks, name introductions, transitions,
  one-word fillers in Q&A, vague pleasantries with no material information.
- "substantive": material numbers, guidance, segment commentary, strategy, specific
  Q&A answers (even short ones if they carry a specific fact, number, or commitment).

ANCHOR EXAMPLES - boilerplate
- "Good afternoon and welcome to the third quarter 2024 earnings conference call."
- "All lines have been placed on mute to prevent any background noise."
- "Today's discussion may include forward-looking statements within the meaning of the Private Securities Litigation Reform Act."
- "Hi, this is Sarah from Goldman Sachs."
- "Thanks for taking my question."
- "I'd add that we're encouraged by the trends." (no specifics)
- "Let me turn the call over to our CFO."
- "That concludes today's call. Thank you for joining."

ANCHOR EXAMPLES - substantive
- "Revenue grew 14% year-over-year to $2.3 billion, driven by strength in our cloud segment."
- "We are raising our full-year EPS guidance to a range of $4.20 to $4.30."
- "Operating margin contracted 80 basis points due to higher input costs."
- "We expect mid-single-digit growth in our consumer business next quarter."
- "We repurchased $500 million of stock during the quarter."
- "We saw strength across all three verticals." (segment commentary even if vague)

EDGE CASES
- One-word answers ("Yes.", "Sure.") with no following content -> boilerplate.
- Mixed sentences ("Hi John, thanks - to your point, margins compressed 60 bps.") ->
  substantive (any material content tips it substantive).
- Hedging without numbers ("We feel good about the trajectory.") -> boilerplate.
- Generic strategy with no specifics ("We continue to invest in innovation.") -> boilerplate.

OUTPUT FORMAT
Return ONLY a single JSON object on one line, no prose, no markdown fences:
{"label": "boilerplate"|"substantive", "confidence": 0.0-1.0, "rationale": "<=15 words"}
"""

# Prompt caching only takes effect above Anthropic's minimum cacheable prompt size.
# These repeated calibration examples are static, so they are cheap after cache warm-up.
_CACHE_PADDING_EXAMPLES = [
    'Boilerplate: "Operator, please open the line for questions." - call logistics only.',
    'Boilerplate: "Thank you, everyone, for joining us today." - closing thanks only.',
    'Boilerplate: "Please note that our remarks contain forward-looking statements." - safe harbor.',
    'Boilerplate: "This is Mark from JPMorgan." - speaker introduction only.',
    'Boilerplate: "We appreciate the question and the continued support." - pleasantry only.',
    'Substantive: "Data center revenue increased 80% year over year to $2.3 billion." - metric plus segment.',
    'Substantive: "We expect gross margin to improve by roughly 50 basis points next quarter." - guidance.',
    'Substantive: "Inventory declined by $120 million as sell-through improved." - financial detail.',
    'Substantive: "Enterprise demand was strongest in healthcare and financial services." - segment commentary.',
    'Substantive: "We signed three hyperscaler customers for the new accelerator platform." - concrete customer detail.',
    'Rule reminder: label the current sentence only, not surrounding transcript context.',
    'Rule reminder: any specific number, guidance range, named segment, or material causal explanation usually makes the sentence substantive.',
]
CACHE_PADDING = "\n".join(
    f"CALIBRATION {rep + 1:02d}.{i + 1:02d}: {example}"
    for rep in range(10)
    for i, example in enumerate(_CACHE_PADDING_EXAMPLES)
)

RUBRIC_BALANCED = RUBRIC_DEFINITIONS + """
You are the BALANCED REVIEWER. Apply the rubric literally. When genuinely on the fence,
report confidence <= 0.55 and pick the class the rubric definitions favor.
""" + CACHE_PADDING

RUBRIC_SKEPTIC = RUBRIC_DEFINITIONS + """
You are the MATERIALITY SKEPTIC. Default to "boilerplate" unless the sentence carries
specific material information (numbers, guidance, named segments/products/strategy)
that a downstream financial analyst would actually want to keep. Vague optimism and
generic strategy talk are boilerplate.
""" + CACHE_PADDING

RUBRIC_PATTERN = RUBRIC_DEFINITIONS + """
You are the PATTERN DETECTOR. Be fast and consistent. Lexical cues like operator
phrases, safe-harbor language, analyst firm names, and generic thanks are strong
boilerplate signals. Numbers, percentages, dollar amounts, basis points, and explicit
guidance language are strong substantive signals.
""" + CACHE_PADDING

JUDGE_RUBRICS = {
    "sonnet_balanced": RUBRIC_BALANCED,
    "sonnet_skeptic":  RUBRIC_SKEPTIC,
    "haiku_pattern":   RUBRIC_PATTERN,
}
print("Rubrics defined:", list(JUDGE_RUBRICS.keys()))
print("Approx rubric chars:", {k: len(v) for k, v in JUDGE_RUBRICS.items()})


Rubrics defined: ['sonnet_balanced', 'sonnet_skeptic', 'haiku_pattern']
Approx rubric chars: {'sonnet_balanced': 16265, 'sonnet_skeptic': 16395, 'haiku_pattern': 16400}


In [7]:
# ---- Anthropic client + per-sentence judge call --------------------------
# Uses prompt caching on the long, repeated rubric and aborts cleanly on hard failures.
from anthropic import Anthropic

_anthropic_client = None


# --- Hard-fail conditions -------------------------------------------------
# These raise immediately and are caught by the labeling loop's `aborted` path
# so we never silently produce hundreds of "FAIL: ..." rows.
class InsufficientBalance(Exception):
    """Account is out of credits. Top up at console.anthropic.com -> Billing."""


class NoApiKey(Exception):
    """ANTHROPIC_API_KEY is unset or still set to the placeholder."""


def _is_balance_error(err: Exception) -> bool:
    msg = (str(err) + " " + repr(err)).lower()
    return ("credit balance" in msg
            or "insufficient_balance" in msg
            or ("billing" in msg and "low" in msg))


def get_client() -> Anthropic:
    """Lazily build the SDK client from the current ANTHROPIC_API_KEY env var."""
    global _anthropic_client
    if _anthropic_client is None:
        key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
        if not key or key == "PASTE_YOUR_KEY_HERE" or not key.startswith("sk-ant-"):
            raise NoApiKey("ANTHROPIC_API_KEY is not set. Set it via "
                           "`os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'` "
                           "and re-run, or use switch_api_key().")
        _anthropic_client = Anthropic(api_key=key)
    return _anthropic_client


def switch_api_key(new_key: str) -> None:
    """Hot-swap the API key mid-session without changing cached votes."""
    global _anthropic_client
    os.environ["ANTHROPIC_API_KEY"] = new_key.strip()
    _anthropic_client = None
    print("API key updated. The next API call will use the new key.")


_JSON_RE = re.compile(r"\{.*?\}", re.DOTALL)


def _parse_judge_json(raw: str) -> dict | None:
    raw = raw.strip()
    for candidate in (raw, *(_JSON_RE.findall(raw) or [])):
        try:
            obj = json.loads(candidate)
        except Exception:
            continue
        label = str(obj.get("label", "")).lower().strip()
        if label not in {"boilerplate", "substantive"}:
            continue
        try:
            conf = float(obj.get("confidence", 0.5))
        except Exception:
            conf = 0.5
        conf = max(0.0, min(1.0, conf))
        return {"label": label, "confidence": conf,
                "rationale": str(obj.get("rationale", ""))[:200]}
    return None


def _response_usage(resp) -> dict:
    usage = getattr(resp, "usage", None)
    fields = [
        "input_tokens", "cache_creation_input_tokens", "cache_read_input_tokens",
        "output_tokens", "server_tool_use", "service_tier",
    ]
    return {f"usage_{name}": getattr(usage, name, None) for name in fields}


def call_judge(sentence: str, judge_name: str, max_retries: int = 4) -> dict:
    """Single judge call with prompt caching and exponential backoff.

    Hard failures (no API key, insufficient balance) RAISE. Ordinary transient
    failures retry a few times; if all retries fail, the caller records a FAIL row.
    """
    model_id = JUDGE_MODELS[judge_name]
    rubric   = JUDGE_RUBRICS[judge_name]
    user_msg = f"SENTENCE:\n{sentence}\n\nReturn the JSON now."

    last_err = None
    for attempt in range(max_retries):
        try:
            resp = get_client().messages.create(
                model=model_id,
                max_tokens=120,
                temperature=0.0,
                system=[{"type": "text", "text": rubric,
                         "cache_control": {"type": "ephemeral"}}],
                messages=[{"role": "user", "content": user_msg}],
            )
            text = "".join(b.text for b in resp.content
                           if getattr(b, "type", "") == "text")
            parsed = _parse_judge_json(text)
            if parsed is not None:
                parsed.update(_response_usage(resp))
                return parsed
            last_err = f"unparseable: {text[:120]!r}"
        except NoApiKey:
            raise
        except Exception as e:
            if _is_balance_error(e):
                raise InsufficientBalance(str(e)) from e
            last_err = repr(e)
        time.sleep(1.5 * (2 ** attempt))       # 1.5, 3, 6, 12 sec

    return {"label": "boilerplate", "confidence": 0.5,
            "rationale": f"FAIL: {last_err}", "_error": True}


In [8]:
#UNCOMMENT if it's your first time running the notebook or if you want to switch API keys without restarting the kernel.
# # ---- API key prompt -----------------------------------------------------
# # Paste your Anthropic key when prompted. The input is hidden and is not saved
# # into this notebook file.
# from getpass import getpass

# _current_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
# if not _current_key.startswith("sk-ant-") or _current_key == "PASTE_YOUR_KEY_HERE":
#     switch_api_key(getpass("Enter your Anthropic API key: "))
# else:
#     print("ANTHROPIC_API_KEY is already set for this kernel.")


In [9]:
# ---- Run all three judges with caching + parallelism --------------------
JUDGE_VOTES_PARQUET = CACHE_DIR / "judge_votes.parquet"
FLUSH_EVERY = 25       # save to disk every N votes (was 100; lower = safer resume)
USAGE_COLS = [
    "usage_input_tokens", "usage_cache_creation_input_tokens",
    "usage_cache_read_input_tokens", "usage_output_tokens",
    "usage_server_tool_use", "usage_service_tier",
]
VOTE_COLS = ["sentence_id", "judge", "label", "confidence", "rationale", *USAGE_COLS]


def _clean_votes(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=VOTE_COLS)
    cleaned = df.copy()
    for col in VOTE_COLS:
        if col not in cleaned.columns:
            cleaned[col] = np.nan
    bad = (
        cleaned["rationale"].astype(str).str.startswith("FAIL:")
        | ~cleaned["label"].isin(["boilerplate", "substantive"])
        | cleaned[["sentence_id", "judge"]].isna().any(axis=1)
    )
    if bad.any():
        print(f"Dropping {bad.sum():,} bad cached vote rows before resume.")
    cleaned = cleaned.loc[~bad, VOTE_COLS]
    return cleaned.drop_duplicates(subset=["sentence_id", "judge"], keep="last")


def load_existing_votes() -> pd.DataFrame:
    if JUDGE_VOTES_PARQUET.exists():
        existing = _clean_votes(pd.read_parquet(JUDGE_VOTES_PARQUET))
        existing.to_parquet(JUDGE_VOTES_PARQUET, index=False)
        return existing
    return pd.DataFrame(columns=VOTE_COLS)


def _vote_row(sid: str, judge: str, res: dict) -> dict:
    row = {
        "sentence_id": sid, "judge": judge,
        "label": res["label"], "confidence": res["confidence"],
        "rationale": res.get("rationale", ""),
    }
    for col in USAGE_COLS:
        row[col] = res.get(col)
    return row


def label_pool(pool: pd.DataFrame, workers: int = PARALLEL_JUDGE_WORKERS) -> pd.DataFrame:
    existing = load_existing_votes()
    have = set(zip(existing["sentence_id"], existing["judge"]))

    todo = []
    for _, row in pool.iterrows():
        for judge in JUDGE_MODELS:
            if (row["sentence_id"], judge) not in have:
                todo.append((row["sentence_id"], row["sentence"], judge))

    if not todo:
        print("All judge votes are cached.")
        return existing

    # Pre-flight: fail fast if the key is missing, before spawning any workers.
    try:
        get_client()
    except NoApiKey as e:
        print(f"!! {e}")
        print("   No API calls were made. Set the key and re-run.")
        return existing

    print(f"Calling judges for {len(todo):,} (sentence, judge) pairs "
          f"with prompt caching enabled...")
    new_rows: list[dict] = []
    aborted = False

    # Warm each distinct judge once, sequentially. Anthropic cache entries become
    # available only after the first response starts; warming avoids the first
    # parallel wave missing the cache for every worker.
    warmup, remaining, warmed = [], [], set()
    for item in todo:
        judge = item[2]
        if judge not in warmed:
            warmup.append(item)
            warmed.add(judge)
        else:
            remaining.append(item)

    try:
        if warmup:
            print(f"Warming {len(warmup)} judge cache prefixes sequentially...")
            for sid, sentence, judge in warmup:
                res = call_judge(sentence, judge)
                new_rows.append(_vote_row(sid, judge, res))
            existing = _flush_votes(existing, new_rows)

        if remaining:
            with ThreadPoolExecutor(max_workers=workers) as ex:
                futures = {
                    ex.submit(call_judge, sentence, judge): (sid, judge)
                    for sid, sentence, judge in remaining
                }
                for fut in tqdm(as_completed(futures), total=len(futures)):
                    sid, judge = futures[fut]
                    try:
                        res = fut.result()
                    except InsufficientBalance as e:
                        print(f"\n!! Insufficient credits: {e}")
                        print("   Stopping cleanly. Cached votes so far are safe on disk.")
                        print("   Top up at console.anthropic.com -> Settings -> Billing,")
                        print("   then re-run this cell - it will resume from where it stopped.")
                        aborted = True
                        break
                    except NoApiKey as e:
                        print(f"\n!! {e}")
                        print("   Stopping. Set the key and re-run this cell.")
                        aborted = True
                        break
                    except Exception as e:
                        res = {"label": "boilerplate", "confidence": 0.5,
                               "rationale": f"FAIL: {e!r}", "_error": True}
                    new_rows.append(_vote_row(sid, judge, res))
                    if len(new_rows) % FLUSH_EVERY == 0:
                        existing = _flush_votes(existing, new_rows)
                if aborted:
                    for f in futures:
                        f.cancel()
    except KeyboardInterrupt:
        print("\n!! Interrupted. Flushing what we have...")
        aborted = True
    except (InsufficientBalance, NoApiKey) as e:
        print(f"\n!! {e}")
        aborted = True

    existing = _flush_votes(existing, new_rows)
    if aborted:
        print("Run stopped early; cached valid votes are preserved.")
    return existing


def _flush_votes(existing: pd.DataFrame, new_rows: list[dict]) -> pd.DataFrame:
    if not new_rows:
        return existing
    combined = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
    combined = _clean_votes(combined)
    combined.to_parquet(JUDGE_VOTES_PARQUET, index=False)
    new_rows.clear()
    return combined


# --- Run the judges -------------------------------------------------------
# To swap to a different API key without restarting the kernel:
#   switch_api_key("sk-ant-...")
votes_df = label_pool(gold_pool)
print(f"Total valid votes on disk: {len(votes_df):,}")
usage_cols_present = [c for c in USAGE_COLS if c in votes_df]
if usage_cols_present and len(votes_df):
    print(votes_df.groupby("judge")[usage_cols_present].sum(numeric_only=True).round(0))
votes_df.head(6)


All judge votes are cached.
Total valid votes on disk: 4,500
                 usage_input_tokens  usage_cache_creation_input_tokens  \
judge                                                                    
haiku_pattern               65780.0                            21630.0   
sonnet_balanced             65780.0                            43020.0   
sonnet_skeptic              65780.0                            43260.0   

                 usage_cache_read_input_tokens  usage_output_tokens  
judge                                                                
haiku_pattern                        6467370.0              65255.0  
sonnet_balanced                      6409980.0              59613.0  
sonnet_skeptic                       6445740.0              59709.0  


,sentence_id,judge,label,confidence,rationale,usage_input_tokens,usage_cache_creation_input_tokens,usage_cache_read_input_tokens,usage_output_tokens,usage_server_tool_use,usage_service_tier
0,a5c9476fbedc,sonnet_balanced,boilerplate,0.85,"Vague pleasantry about customer engagements, n...",32.0,0.0,4302.0,43.0,None,standard
1,a5c9476fbedc,sonnet_skeptic,boilerplate,0.95,"Vague optimism about customer engagements, no ...",32.0,0.0,4326.0,42.0,None,standard
2,a5c9476fbedc,haiku_pattern,boilerplate,0.95,"Generic pleasantry with no specifics, numbers,...",32.0,0.0,4326.0,40.0,None,standard
3,cd7521ab9a17,haiku_pattern,boilerplate,0.92,Generic forward-looking statement with no spec...,28.0,0.0,4326.0,42.0,None,standard
4,e6a7bb29a86c,haiku_pattern,boilerplate,0.92,Vague hedging statement with no specific numbe...,29.0,0.0,4326.0,42.0,None,standard
5,e6a7bb29a86c,sonnet_balanced,substantive,0.52,"References gross margin mix impact, a material...",29.0,0.0,4302.0,39.0,None,standard


In [10]:
# ---- Top up missing judge votes ----------------------------------------
# Run this after the main judge cell if it finished but Cell 13 says some
# valid votes are missing. This reads the parquet cache and calls only the
# missing (sentence_id, judge) pairs; already-saved votes are not repeated.
def missing_vote_pairs(pool: pd.DataFrame, votes: pd.DataFrame) -> list[tuple[str, str, str]]:
    have = set(zip(votes["sentence_id"], votes["judge"])) if len(votes) else set()
    missing = []
    for _, row in pool.iterrows():
        for judge in JUDGE_MODELS:
            if (row["sentence_id"], judge) not in have:
                missing.append((row["sentence_id"], row["sentence"], judge))
    return missing


def top_up_missing_votes(pool: pd.DataFrame, workers: int = PARALLEL_JUDGE_WORKERS,
                         passes: int = 3) -> pd.DataFrame:
    existing = load_existing_votes()
    expected = len(pool) * len(JUDGE_MODELS)

    for pass_no in range(1, passes + 1):
        todo = missing_vote_pairs(pool, existing)
        if not todo:
            print(f"All judge votes are present: {len(existing):,}/{expected:,}.")
            return existing

        print(f"Top-up pass {pass_no}/{passes}: calling {len(todo):,} missing pairs only...")
        new_rows: list[dict] = []
        aborted = False

        try:
            get_client()
            with ThreadPoolExecutor(max_workers=workers) as ex:
                futures = {
                    ex.submit(call_judge, sentence, judge): (sid, judge)
                    for sid, sentence, judge in todo
                }
                for fut in tqdm(as_completed(futures), total=len(futures)):
                    sid, judge = futures[fut]
                    try:
                        res = fut.result()
                    except InsufficientBalance as e:
                        print(f"Insufficient credits: {e}")
                        aborted = True
                        break
                    except NoApiKey as e:
                        print(f"{e}")
                        aborted = True
                        break
                    except Exception as e:
                        # Do not save transient failures as labels. Leave them missing
                        # so another top-up pass can retry them.
                        print(f"Transient failure for {sid}/{judge}: {e!r}")
                        continue

                    if str(res.get("rationale", "")).startswith("FAIL:"):
                        continue
                    new_rows.append(_vote_row(sid, judge, res))
                    if len(new_rows) % FLUSH_EVERY == 0:
                        existing = _flush_votes(existing, new_rows)

                if aborted:
                    for f in futures:
                        f.cancel()
                    break
        except KeyboardInterrupt:
            print("Interrupted. Flushing completed top-up votes...")
            break
        finally:
            existing = _flush_votes(existing, new_rows)
            print(f"Valid votes now: {len(existing):,}/{expected:,}")

    remaining = len(missing_vote_pairs(pool, existing))
    if remaining:
        print(f"Still missing {remaining:,} votes. Re-run this top-up cell to continue.")
    return existing


votes_df = top_up_missing_votes(gold_pool)
votes_df.head(6)


All judge votes are present: 4,500/4,500.


,sentence_id,judge,label,confidence,rationale,usage_input_tokens,usage_cache_creation_input_tokens,usage_cache_read_input_tokens,usage_output_tokens,usage_server_tool_use,usage_service_tier
0,a5c9476fbedc,sonnet_balanced,boilerplate,0.85,"Vague pleasantry about customer engagements, n...",32.0,0.0,4302.0,43.0,None,standard
1,a5c9476fbedc,sonnet_skeptic,boilerplate,0.95,"Vague optimism about customer engagements, no ...",32.0,0.0,4326.0,42.0,None,standard
2,a5c9476fbedc,haiku_pattern,boilerplate,0.95,"Generic pleasantry with no specifics, numbers,...",32.0,0.0,4326.0,40.0,None,standard
3,cd7521ab9a17,haiku_pattern,boilerplate,0.92,Generic forward-looking statement with no spec...,28.0,0.0,4326.0,42.0,None,standard
4,e6a7bb29a86c,haiku_pattern,boilerplate,0.92,Vague hedging statement with no specific numbe...,29.0,0.0,4326.0,42.0,None,standard
5,e6a7bb29a86c,sonnet_balanced,substantive,0.52,"References gross margin mix impact, a material...",29.0,0.0,4302.0,39.0,None,standard


## 3 · Adjudication, audit, and freezing the gold set

Compute the majority label and report:
- pairwise agreement rates between judges,
- overall full-agreement rate (all three agree),
- distribution of disagreements by direction,
- a stratified audit sample for human review (`reports/disagreement_audit.csv`),
- final class balance.

The handout requires both per-judge agreement reporting and an audit; both live below.


In [11]:
def adjudicate(votes: pd.DataFrame) -> pd.DataFrame:
    wide = votes.pivot_table(
        index="sentence_id", columns="judge",
        values=["label", "confidence"], aggfunc="first",
    )
    wide.columns = [f"{c1}__{c0}" for c0, c1 in wide.columns]  # judge__label / judge__confidence
    wide = wide.reset_index()

    label_cols = [f"{j}__label" for j in JUDGE_MODELS]
    for col in label_cols:
        if col not in wide.columns:
            wide[col] = np.nan

    def majority(row):
        labs = [row[c] for c in label_cols if isinstance(row[c], str)]
        if len(labs) < len(JUDGE_MODELS):
            return (np.nan, len(labs))
        n_sub = sum(1 for l in labs if l == "substantive")
        n_boi = len(labs) - n_sub
        winner = "substantive" if n_sub > n_boi else "boilerplate"
        return (winner, max(n_sub, n_boi))

    wide[["gold_label", "vote_count"]] = wide.apply(
        lambda r: pd.Series(majority(r)), axis=1
    )
    wide["full_agreement"] = wide.apply(
        lambda r: len({r[c] for c in label_cols if isinstance(r[c], str)}) == len(JUDGE_MODELS), axis=1
    )
    return wide


expected_votes = len(gold_pool) * len(JUDGE_MODELS)
fail_votes = votes_df["rationale"].astype(str).str.startswith("FAIL:").sum() if len(votes_df) else 0
if fail_votes:
    raise RuntimeError(f"Found {fail_votes:,} FAIL rows in votes_df. Clean/re-run Cell 11 before adjudication.")
if len(votes_df) < expected_votes:
    raise RuntimeError(
        f"Only {len(votes_df):,}/{expected_votes:,} valid judge votes are available. "
        "Let Cell 11 finish or intentionally reduce GOLD_POOL_SIZE and resample before adjudication."
    )

adjudicated = adjudicate(votes_df)
adjudicated = adjudicated.merge(gold_pool[["sentence_id", "sentence", "transcript"]],
                                on="sentence_id", how="left")

# Pairwise agreement
judges = list(JUDGE_MODELS)
print("\nPairwise label agreement:")
for i in range(len(judges)):
    for j in range(i + 1, len(judges)):
        a, b = judges[i], judges[j]
        agree = (adjudicated[f"{a}__label"] == adjudicated[f"{b}__label"]).mean()
        print(f"  {a:18s} vs {b:18s}  {agree:.3f}")

print(f"\nFull-agreement rate (all 3 judges agree): {adjudicated['full_agreement'].mean():.3f}")
print(f"Disagreement rate: {1 - adjudicated['full_agreement'].mean():.3f}")
print("\nMajority-vote class balance:")
print(adjudicated["gold_label"].value_counts(normalize=True).round(3))



Pairwise label agreement:
  sonnet_balanced    vs sonnet_skeptic      0.938
  sonnet_balanced    vs haiku_pattern       0.873
  sonnet_skeptic     vs haiku_pattern       0.902

Full-agreement rate (all 3 judges agree): 0.000
Disagreement rate: 1.000

Majority-vote class balance:
gold_label
substantive    0.525
boilerplate    0.475
Name: proportion, dtype: float64


In [12]:
# ---- Stratified audit sample for human review --------------------------
disagreed = adjudicated.loc[~adjudicated["full_agreement"]].copy()

# Stratify by direction of disagreement (substantive-leaning vs boilerplate-leaning)
disagreed["direction"] = np.where(
    disagreed["vote_count"] == 2,
    disagreed["gold_label"] + "_2v1",
    "split",
)

audit_n = min(80, len(disagreed))
audit = (disagreed.groupby("direction", group_keys=False)
                   .apply(lambda g: g.sample(min(len(g), audit_n // max(1, disagreed["direction"].nunique())),
                                             random_state=SEED)))
audit_path = REPORTS_DIR / "disagreement_audit.csv"
audit_cols = (["sentence_id", "transcript", "sentence", "gold_label", "vote_count", "direction"]
              + [f"{j}__label" for j in judges]
              + [f"{j}__confidence" for j in judges])
audit[audit_cols].to_csv(audit_path, index=False)
print(f"Wrote {len(audit)} disagreement cases to {audit_path}")
print("→ Open the CSV, override `gold_label` for any obvious mistakes, save, then run the next cell.")
audit[audit_cols].head(8)


Wrote 78 disagreement cases to /Users/chaithanyapakala/Documents/NLP/Pakala_Chaithanya_NLP_HW2/reports/disagreement_audit.csv
→ Open the CSV, override `gold_label` for any obvious mistakes, save, then run the next cell.


,sentence_id,transcript,sentence,gold_label,vote_count,direction,sonnet_balanced__label,sonnet_skeptic__label,haiku_pattern__label,sonnet_balanced__confidence,sonnet_skeptic__confidence,haiku_pattern__confidence
630,65321d37b210,PLTR_Q2-2025,"As LLMs continue to improve, it only further a...",boilerplate,2,boilerplate_2v1,substantive,boilerplate,boilerplate,0.55,0.82,0.85
5,013c18ba28ea,C_Q3-2024,The U.S. consumer dynamics remain remarkably c...,boilerplate,2,boilerplate_2v1,substantive,boilerplate,boilerplate,0.55,0.82,0.85
491,4d5b04cafa7a,INTC_Q4-2024,"As such, a one-size-fits-all approach will not...",boilerplate,2,boilerplate_2v1,substantive,boilerplate,boilerplate,0.52,0.85,0.92
632,657033f058e0,WFC_Q3-2025,Analysts - MD & Head of United States Bank Res...,boilerplate,2,boilerplate_2v1,boilerplate,boilerplate,substantive,0.72,0.85,0.85
406,3ff760e939cf,FAST_Q2-2025,But what you saw was an incredible leaning dow...,boilerplate,2,boilerplate_2v1,substantive,boilerplate,boilerplate,0.55,0.72,0.85
621,6318a962a694,NVDA_Q3-2026,"Therefore, a new approach is necessary for the...",boilerplate,2,boilerplate_2v1,substantive,boilerplate,boilerplate,0.45,0.75,0.92
216,21df585fc180,GS_Q3-2024,The final rule will have a significant impact ...,boilerplate,2,boilerplate_2v1,substantive,boilerplate,boilerplate,0.52,0.82,0.92
1353,e5e8ca998cbd,AVGO_Q2-2025,And do you think it leads to a market share sh...,boilerplate,2,boilerplate_2v1,substantive,boilerplate,boilerplate,0.60,0.72,0.92


In [13]:
# ---- Reload the audit (with any human corrections) and freeze the gold ------
audit_path = REPORTS_DIR / "disagreement_audit.csv"
if audit_path.exists():
    audited = pd.read_csv(audit_path)
    overrides = audited.set_index("sentence_id")["gold_label"].to_dict()
    n_changed = sum(1 for sid, lab in overrides.items()
                    if sid in adjudicated["sentence_id"].values
                    and adjudicated.loc[adjudicated["sentence_id"] == sid, "gold_label"].iloc[0] != lab)
    if n_changed:
        adjudicated["gold_label"] = adjudicated.apply(
            lambda r: overrides.get(r["sentence_id"], r["gold_label"]), axis=1
        )
        print(f"Applied {n_changed} human overrides from the audit CSV.")
    else:
        print("No overrides detected — keeping pure majority vote.")

GOLD_PARQUET = CACHE_DIR / "gold.parquet"
gold = adjudicated[["sentence_id", "transcript", "sentence", "gold_label",
                    "full_agreement", "vote_count"]].copy()
gold["y"] = (gold["gold_label"] == "substantive").astype(int)   # 1 = substantive (positive class)
gold.to_parquet(GOLD_PARQUET, index=False)

print(f"\\nFrozen gold set: {len(gold):,} sentences")
print(gold["gold_label"].value_counts())
print(f"Full-agreement share: {gold['full_agreement'].mean():.3f}")


No overrides detected — keeping pure majority vote.
\nFrozen gold set: 1,500 sentences
gold_label
substantive    788
boilerplate    712
Name: count, dtype: int64
Full-agreement share: 0.000


## 4 · Stratified 60 / 20 / 20 split

The test split is **frozen**: nothing downstream touches it until the final test cell.


In [14]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    gold, test_size=0.40, stratify=gold["y"], random_state=SEED,
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["y"], random_state=SEED,
)
for name, d in (("train", train_df), ("val", val_df), ("test", test_df)):
    d.to_parquet(CACHE_DIR / f"split_{name}.parquet", index=False)
    pos = d["y"].mean()
    print(f"{name:5s}  n={len(d):4d}  substantive={pos:.3f}  boilerplate={1-pos:.3f}")


train  n= 900  substantive=0.526  boilerplate=0.474
val    n= 300  substantive=0.527  boilerplate=0.473
test   n= 300  substantive=0.523  boilerplate=0.477


## 5 · Feature engineering

Two families: ~30 hand-crafted regex flags + frozen MPNet embeddings (768-d). Embeddings are cached.


In [15]:
# ---- Regex feature pack -------------------------------------------------
FEATURE_PATTERNS = [
    # OPERATOR / HOUSEKEEPING (boilerplate signals)
    ("starts_with_operator", r"^\s*(operator|moderator)[\s:.,]"),
    ("mute_lines",           r"\b(lines? (have been|are) placed on mute|on mute|in listen[- ]only mode)\b"),
    ("queue_phrase",         r"\b(press\s*\*?\s*[1-9]|press star|in the queue|withdraw your question|ask a question)\b"),
    ("recording_phrase",     r"\b(call is being recorded|today's call is being recorded)\b"),

    # WELCOME / CLOSING
    ("welcome_phrase",       r"\b(welcome to (the|today's|our)|good (morning|afternoon|evening),?\s*(everyone|all|ladies))"),
    ("conclude_phrase",      r"\b(this concludes|thank you for (joining|participating)|that concludes our call)"),
    ("turn_call_over",       r"\b(turn (the call|it) (over|back) to|hand (it|the call) (over|back) to)"),

    # SAFE HARBOR / FORWARD LOOKING
    ("forward_looking",      r"\bforward[- ]looking statements?\b"),
    ("safe_harbor",          r"\b(safe harbor|private securities litigation reform act|may differ materially|risks and uncertainties)\b"),
    ("non_gaap",             r"\b(non[- ]?gaap|gaap (to|and non)|reconciliation (of|to) gaap)\b"),
    ("sec_filings",          r"\b(10[- ]?[KQ]|sec (filings?|filing)|annual report|proxy statement)\b"),

    # Q&A PLEASANTRIES
    ("thanks_for_question",  r"\b(thanks?|thank you)( so much)?( ,)? for (taking|having|the question|joining)"),
    ("generic_greeting",     r"^\s*(hi|hey|hello|good (morning|afternoon|evening))[, ]"),
    ("name_intro",           r"\bthis is \w+ (from|at|with|on for)\s+[A-Z]\w+"),
    ("analyst_firm",         r"\b(goldman( sachs)?|jp\s*morgan|morgan stanley|wells fargo|bank of america|citi(group)?|barclays|deutsche|ubs|credit suisse|jefferies|cowen|raymond james|piper sandler|wedbush|kbw|stifel|baird|evercore|guggenheim|bernstein|oppenheimer|truist|td (cowen|securities))\b"),

    # MATERIAL CONTENT (substantive signals)
    ("has_dollar",           r"\$\s?\d"),
    ("has_percent",          r"\d+(\.\d+)?\s?%|\bpercent\b"),
    ("has_bps",              r"\bbasis points?\b|\bbps\b"),
    ("has_million_billion",  r"\b(million|billion|trillion|mn|bn)\b"),
    ("has_year",             r"\b(20[12]\d|fiscal\s+\d{4}|fy\s?\d{4})\b"),
    ("has_quarter",          r"\b(q[1-4]\b|first quarter|second quarter|third quarter|fourth quarter|fy\d+q\d)\b"),

    # GUIDANCE / STRATEGY
    ("guidance_word",        r"\b(guidance|guide|outlook|expect|anticipate|forecast|raise|raising|raised|reaffirm|reiterate)\b"),
    ("segment_word",         r"\b(segment|division|business unit|product line|geography|vertical|category)\b"),
    ("margin_word",          r"\b(margin|operating income|gross margin|EBITDA|free cash flow|FCF|EPS|earnings per share)\b"),
]

_COMPILED = [(name, re.compile(pat, re.IGNORECASE)) for name, pat in FEATURE_PATTERNS]


def regex_features(sentence: str) -> dict:
    s = sentence
    feats = {name: int(bool(rx.search(s))) for name, rx in _COMPILED}

    # Lexical / structural (continuous)
    n_chars = len(s)
    words = s.split()
    n_words = len(words)
    n_digits = sum(c.isdigit() for c in s)
    n_alpha  = sum(c.isalpha() for c in s)
    n_upper  = sum(c.isupper() for c in s)

    feats.update({
        "len_chars":      n_chars,
        "len_words":      n_words,
        "digit_ratio":    n_digits / max(1, n_chars),
        "uppercase_ratio": n_upper / max(1, n_alpha),
        "ends_with_question": int(s.rstrip().endswith("?")),
        "starts_with_number": int(bool(re.match(r"^\s*\d", s))),
        "first_person_count": sum(1 for w in words if w.lower() in {"we", "our", "us", "i"}),
        "modal_count":     sum(1 for w in words if w.lower() in
                               {"will", "would", "expect", "expects", "expected",
                                "plan", "plans", "intend", "may", "might", "could", "should"}),
        "proper_noun_run": int(bool(re.search(r"\b([A-Z][a-z]+\s+){2,}", s))),
    })
    return feats


REGEX_FEATURE_NAMES = list(regex_features("placeholder").keys())
print(f"{len(REGEX_FEATURE_NAMES)} regex features:")
print(", ".join(REGEX_FEATURE_NAMES))


33 regex features:
starts_with_operator, mute_lines, queue_phrase, recording_phrase, welcome_phrase, conclude_phrase, turn_call_over, forward_looking, safe_harbor, non_gaap, sec_filings, thanks_for_question, generic_greeting, name_intro, analyst_firm, has_dollar, has_percent, has_bps, has_million_billion, has_year, has_quarter, guidance_word, segment_word, margin_word, len_chars, len_words, digit_ratio, uppercase_ratio, ends_with_question, starts_with_number, first_person_count, modal_count, proper_noun_run


In [16]:
# ---- Build regex feature matrices for all three splits -------------------
def regex_matrix(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame([regex_features(s) for s in df["sentence"]],
                        index=df.index, columns=REGEX_FEATURE_NAMES)

X_train_rx = regex_matrix(train_df)
X_val_rx   = regex_matrix(val_df)
X_test_rx  = regex_matrix(test_df)
y_train = train_df["y"].values
y_val   = val_df["y"].values
y_test  = test_df["y"].values

print("Regex matrix shapes:",
      X_train_rx.shape, X_val_rx.shape, X_test_rx.shape)


Regex matrix shapes: (900, 33) (300, 33) (300, 33)


In [17]:
# ---- Frozen sentence embeddings ----------------------------------------
EMBED_CACHE = CACHE_DIR / f"embeddings_{EMBED_MODEL_NAME.replace('/', '__')}.npz"

def embed_sentences(sentences: list[str]) -> np.ndarray:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(EMBED_MODEL_NAME)
    return model.encode(sentences, batch_size=64, show_progress_bar=True,
                        convert_to_numpy=True, normalize_embeddings=True)


if EMBED_CACHE.exists():
    cache = np.load(EMBED_CACHE, allow_pickle=True)
    cached_ids = set(cache["sentence_ids"].tolist())
    cached_emb = {sid: cache["embeddings"][i] for i, sid in enumerate(cache["sentence_ids"])}
else:
    cached_ids, cached_emb = set(), {}

needed_df = gold[~gold["sentence_id"].isin(cached_ids)]
if len(needed_df):
    print(f"Embedding {len(needed_df):,} new sentences...")
    new_emb = embed_sentences(needed_df["sentence"].tolist())
    for sid, emb in zip(needed_df["sentence_id"], new_emb):
        cached_emb[sid] = emb
    sids = np.array(list(cached_emb.keys()))
    embs = np.stack([cached_emb[sid] for sid in sids])
    np.savez_compressed(EMBED_CACHE, sentence_ids=sids, embeddings=embs)

def get_emb(df: pd.DataFrame) -> np.ndarray:
    return np.stack([cached_emb[sid] for sid in df["sentence_id"]])

X_train_emb = get_emb(train_df)
X_val_emb   = get_emb(val_df)
X_test_emb  = get_emb(test_df)

X_train_full = np.hstack([X_train_emb, X_train_rx.values])
X_val_full   = np.hstack([X_val_emb,   X_val_rx.values])
X_test_full  = np.hstack([X_test_emb,  X_test_rx.values])

print("Embedding shapes:", X_train_emb.shape, X_val_emb.shape, X_test_emb.shape)
print("Combined  shapes:", X_train_full.shape, X_val_full.shape, X_test_full.shape)


Embedding shapes: (900, 768) (300, 768) (300, 768)
Combined  shapes: (900, 801) (300, 801) (300, 801)


## 6 · Classifier zoo (12 entries, 7 families)

Each entry stores its 5-fold OOF probabilities (used for threshold tuning) and held-out test probabilities (used for the leaderboard).

| # | Family | Entry |
|---|---|---|
| 1 | Rules | Regex-feature majority vote |
| 2 | Linear | LogReg on MPNet embeddings |
| 3 | Linear (lexical) | Linear SVM on TF-IDF char n-grams (calibrated) |
| 4 | Tree | HistGradientBoosting on (embeddings ⊕ regex) |
| 5 | n-gram | FastText supervised |
| 6 | Contrastive | SetFit on MPNet |
| 7 | Transformer | FinBERT fine-tuned |
| 8 | Anchor (creative) | Prototype-cosine classifier |
| 9 | Hybrid (creative) | Two-stage triage (rules → embedding LogReg) |
| 10 | Distillation (creative) | Soft-label LogReg trained on mean-judge probability |
| 11 | Ensemble | Mean-probability of top-5 non-transformer |
| 12 | Ensemble | Stacked meta-LogReg on OOF probabilities |

Entries 8, 9, 10, 12 are the creative additions. Each cell is self-contained: re-run any single one without touching the others.


In [18]:
# ---- Shared infrastructure ----------------------------------------------
from sklearn.metrics import (precision_recall_fscore_support, confusion_matrix,
                             classification_report, f1_score, recall_score)
from sklearn.model_selection import StratifiedKFold


@dataclass
class Entry:
    name: str
    family: str
    description: str
    train_seconds: float = 0.0
    sentences_per_second: float = 0.0
    oof_proba: np.ndarray = field(default_factory=lambda: np.zeros(0))
    test_proba: np.ndarray = field(default_factory=lambda: np.zeros(0))
    notes: str = ""


# Train+val pool for OOF threshold tuning
X_pool_emb  = np.vstack([X_train_emb,  X_val_emb])
X_pool_rx   = pd.concat([X_train_rx, X_val_rx]).reset_index(drop=True)
X_pool_full = np.vstack([X_train_full, X_val_full])
y_pool      = np.concatenate([y_train, y_val])
df_pool     = pd.concat([train_df, val_df]).reset_index(drop=True)

print("Pool size:", len(y_pool), "  test size:", len(y_test))

ENTRIES: dict[str, Entry] = {}

def cv_indices():
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    return list(skf.split(X_pool_emb, y_pool))


def time_inference(predict_fn, X) -> float:
    """Return sentences/second (rough — wall-clock, single thread)."""
    t0 = time.perf_counter()
    predict_fn(X)
    elapsed = time.perf_counter() - t0
    return len(X) / max(elapsed, 1e-9)


Pool size: 1200   test size: 300


In [19]:
# Entry 1 — Rules-only baseline (no learning, threshold via signal-count) -----
def rules_score(rx_df: pd.DataFrame) -> np.ndarray:
    """Score = sigmoid( substantive_signals - boilerplate_signals ). Higher → substantive."""
    boilerplate_keys = ["starts_with_operator", "mute_lines", "queue_phrase",
                        "recording_phrase", "welcome_phrase", "conclude_phrase",
                        "turn_call_over", "forward_looking", "safe_harbor",
                        "thanks_for_question", "generic_greeting", "name_intro",
                        "analyst_firm"]
    substantive_keys = ["has_dollar", "has_percent", "has_bps",
                        "has_million_billion", "has_year", "has_quarter",
                        "guidance_word", "segment_word", "margin_word"]
    boi = rx_df[boilerplate_keys].sum(axis=1).values
    sub = rx_df[substantive_keys].sum(axis=1).values
    z = sub.astype(float) - boi.astype(float)
    return 1.0 / (1.0 + np.exp(-z))                        # sigmoid


t0 = time.perf_counter()
oof = rules_score(X_pool_rx)
t_train = time.perf_counter() - t0

t0 = time.perf_counter()
test_p = rules_score(X_test_rx)
t_pred = time.perf_counter() - t0

ENTRIES["rules_only"] = Entry(
    name="rules_only", family="Rules",
    description="Regex feature signal-count → sigmoid",
    train_seconds=t_train, sentences_per_second=len(X_test_rx) / max(t_pred, 1e-9),
    oof_proba=oof, test_proba=test_p,
    notes="Zero learning. Bound by feature coverage.",
)
print("rules_only OK:", oof.shape, test_p.shape)


rules_only OK: (1200,) (300,)


In [20]:
# Entry 2 — LogReg on MPNet embeddings -------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


def make_logreg_emb():
    return Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced",
                                   random_state=SEED)),
    ])


oof = np.zeros(len(y_pool))
t0 = time.perf_counter()
for tr, te in cv_indices():
    m = make_logreg_emb()
    m.fit(X_pool_emb[tr], y_pool[tr])
    oof[te] = m.predict_proba(X_pool_emb[te])[:, 1]
t_train = time.perf_counter() - t0

final = make_logreg_emb().fit(X_pool_emb, y_pool)
sps = time_inference(lambda X: final.predict_proba(X), X_test_emb)
test_p = final.predict_proba(X_test_emb)[:, 1]

ENTRIES["logreg_embed"] = Entry(
    name="logreg_embed", family="Linear",
    description="LogReg on frozen MPNet embeddings (class-weighted)",
    train_seconds=t_train, sentences_per_second=sps,
    oof_proba=oof, test_proba=test_p,
)
print("logreg_embed OK")


logreg_embed OK


In [21]:
# Entry 3 — Linear SVM (calibrated) on TF-IDF char n-grams -----------------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV


def make_svm_tfidf():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                                  min_df=2, sublinear_tf=True)),
        ("clf", CalibratedClassifierCV(LinearSVC(C=1.0, class_weight="balanced",
                                                 random_state=SEED),
                                       method="sigmoid", cv=3)),
    ])


texts_pool = pd.concat([train_df, val_df])["sentence"].tolist()
texts_test = test_df["sentence"].tolist()

oof = np.zeros(len(y_pool))
t0 = time.perf_counter()
for tr, te in cv_indices():
    m = make_svm_tfidf()
    m.fit([texts_pool[i] for i in tr], y_pool[tr])
    oof[te] = m.predict_proba([texts_pool[i] for i in te])[:, 1]
t_train = time.perf_counter() - t0

final = make_svm_tfidf().fit(texts_pool, y_pool)
sps = time_inference(lambda X: final.predict_proba(X), texts_test)
test_p = final.predict_proba(texts_test)[:, 1]

ENTRIES["svm_charngram"] = Entry(
    name="svm_charngram", family="Linear (lexical)",
    description="Linear SVM on char 3–5 n-grams, sigmoid-calibrated",
    train_seconds=t_train, sentences_per_second=sps,
    oof_proba=oof, test_proba=test_p,
    notes="Different feature space than embeddings — useful ensemble member.",
)
print("svm_charngram OK")


svm_charngram OK


In [22]:
# Entry 4 — HistGradientBoosting on (embeddings + regex flags) -------------
from sklearn.ensemble import HistGradientBoostingClassifier


def make_hgb():
    return HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.07, max_depth=6,
        min_samples_leaf=20, random_state=SEED, class_weight="balanced",
    )


oof = np.zeros(len(y_pool))
t0 = time.perf_counter()
for tr, te in cv_indices():
    m = make_hgb()
    m.fit(X_pool_full[tr], y_pool[tr])
    oof[te] = m.predict_proba(X_pool_full[te])[:, 1]
t_train = time.perf_counter() - t0

final = make_hgb().fit(X_pool_full, y_pool)
sps = time_inference(lambda X: final.predict_proba(X), X_test_full)
test_p = final.predict_proba(X_test_full)[:, 1]

ENTRIES["hgb_combined"] = Entry(
    name="hgb_combined", family="Tree ensemble",
    description="HistGradientBoosting on (embeddings ⊕ 30 regex flags)",
    train_seconds=t_train, sentences_per_second=sps,
    oof_proba=oof, test_proba=test_p,
)
print("hgb_combined OK")


hgb_combined OK


In [23]:
pip install fasttext-wheel


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
# Entry 5 — FastText (supervised) ------------------------------------------
try:
    import fasttext
    HAS_FASTTEXT = True
except Exception:
    HAS_FASTTEXT = False
    print("fasttext not installed — skipping. `pip install fasttext-wheel` to enable.")


if HAS_FASTTEXT:
    def to_ft_lines(texts, labels):
        return [f"__label__{l} {t.replace(chr(10), ' ')}\n" for t, l in zip(texts, labels)]

    def train_ft(texts, labels, path: Path):
        path.write_text("".join(to_ft_lines(texts, labels)))
        return fasttext.train_supervised(
            input=str(path), lr=0.5, epoch=25, wordNgrams=2,
            dim=100, minCount=1, loss="softmax", verbose=0,
        )

    def ft_proba(model, texts) -> np.ndarray:
        out = np.zeros(len(texts))
        for i, t in enumerate(texts):
            clean = t.replace("\n", " ")
            # fasttext's Python wrapper calls np.array(..., copy=False), which
            # breaks under NumPy 2.x. The pybind method returns plain tuples.
            pred = model.f.predict(clean, 2, 0.0, "strict")
            for p, label in pred:
                if label.endswith("__1"):
                    out[i] = float(p)
        return out

    oof = np.zeros(len(y_pool))
    ft_train_path = CACHE_DIR / "ft_train.txt"
    t0 = time.perf_counter()
    for fold, (tr, te) in enumerate(cv_indices()):
        m = train_ft([texts_pool[i] for i in tr], y_pool[tr], ft_train_path)
        oof[te] = ft_proba(m, [texts_pool[i] for i in te])
    t_train = time.perf_counter() - t0

    final = train_ft(texts_pool, y_pool, ft_train_path)
    sps = time_inference(lambda X: ft_proba(final, X), texts_test)
    test_p = ft_proba(final, texts_test)

    ENTRIES["fasttext"] = Entry(
        name="fasttext", family="N-gram",
        description="FastText supervised, wordNgrams=2, dim=100, 25 epochs",
        train_seconds=t_train, sentences_per_second=sps,
        oof_proba=oof, test_proba=test_p,
    )
    print("fasttext OK")


fasttext OK


In [25]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'setfit'], check=True)

# Entry 6 — SetFit (contrastive fine-tuning) -------------------------------
# Compatibility guard: if the live kernel imported Homebrew transformers 5.x
# before we installed the compatible user-site 4.x package, clear the stale
# modules and put the user site first. This avoids a full rerun of prior cells.
import importlib, site, sys, warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"sentence_transformers\.cross_encoder.*")

_user_site = site.getusersitepackages()
if _user_site in sys.path:
    sys.path.remove(_user_site)
sys.path.insert(0, _user_site)
for _mod in list(sys.modules):
    if (_mod == "transformers" or _mod.startswith("transformers.")
            or _mod == "sentence_transformers" or _mod.startswith("sentence_transformers.")
            or _mod == "setfit" or _mod.startswith("setfit.")):
        del sys.modules[_mod]
importlib.invalidate_caches()

if ENABLE_SETFIT:
    try:
        from setfit import SetFitModel, Trainer, TrainingArguments
        from datasets import Dataset
        HAS_SETFIT = True
    except Exception as e:
        HAS_SETFIT = False
        print("SetFit not available:", e)

    if HAS_SETFIT:
        def make_setfit():
            return SetFitModel.from_pretrained(EMBED_MODEL_NAME)

        SETFIT_NUM_ITERATIONS = 4
        SETFIT_MAX_STEPS = 80

        def fit_setfit(texts, labels, run_name="setfit"):
            ds = Dataset.from_dict({"text": list(texts), "label": list(labels)})
            model = make_setfit()
            args = TrainingArguments(
                batch_size=8,
                num_iterations=SETFIT_NUM_ITERATIONS,
                num_epochs=1,
                max_steps=SETFIT_MAX_STEPS,
                output_dir=str(CACHE_DIR / "setfit_out" / run_name),
                report_to="none",
                save_strategy="no",
                show_progress_bar=False,
            )
            Trainer(model=model, args=args, train_dataset=ds).train()
            return model

        oof = np.zeros(len(y_pool))
        t0 = time.perf_counter()
        for fold, (tr, te) in enumerate(cv_indices()):
            print(f"SetFit fold {fold + 1}/{N_FOLDS}...")
            m = fit_setfit([texts_pool[i] for i in tr], y_pool[tr], run_name=f"fold_{fold}")
            probs = np.asarray(m.predict_proba([texts_pool[i] for i in te]))
            oof[te] = (probs[:, 1] if probs.ndim == 2 and probs.shape[1] == 2
                       else probs.reshape(-1))
            del m
        t_train = time.perf_counter() - t0

        print("SetFit final fit...")
        final = fit_setfit(texts_pool, y_pool, run_name="final")
        SETFIT_FINAL = final
        SETFIT_MODEL_DIR = MODELS_DIR / "setfit_final"
        final.save_pretrained(str(SETFIT_MODEL_DIR))
        def setfit_predict(X):
            p = np.asarray(final.predict_proba(list(X)))
            return p[:, 1] if p.ndim == 2 and p.shape[1] == 2 else p.reshape(-1)
        sps = time_inference(setfit_predict, texts_test)
        test_p = setfit_predict(texts_test)

        ENTRIES["setfit"] = Entry(
            name="setfit", family="Contrastive",
            description=f"SetFit on MPNet, {SETFIT_NUM_ITERATIONS} iterations, max {SETFIT_MAX_STEPS} steps",
            train_seconds=t_train, sentences_per_second=sps,
            oof_proba=oof, test_proba=test_p,
            notes="Often the strongest single model for small labeled sets.",
        )
        print("setfit OK")



[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


SetFit fold 1/5...


model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Map: 100%|██████████| 960/960 [00:00<00:00, 46318.71 examples/s]
***** Running training *****
  Num unique pairs = 7680
  Batch size = 8
  Num epochs = 1


Step,Training Loss
1,0.559700
50,0.247000


SetFit fold 2/5...


...

Step,Training Loss
1,0.411300
50,0.247000


SetFit final fit...


model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Map: 100%|██████████| 1200/1200 [00:00<00:00, 45289.56 examples/s]
***** Running training *****
  Num unique pairs = 9600
  Batch size = 8
  Num epochs = 1


Step,Training Loss
1,0.523800
50,0.246500


setfit OK


In [26]:
import requests as _req, transformers.utils.hub as _hub, transformers.modeling_utils as _mutils
from huggingface_hub import hf_hub_url

def _safe_has_file(path_or_repo, filename, revision=None, proxies=None, token=None,
                   local_files_only=False, cache_dir=None, repo_type='model', **kw):
    if local_files_only: return False
    try:
        url = hf_hub_url(path_or_repo, filename=filename, revision=revision, repo_type=repo_type)
        return _req.head(url, allow_redirects=False, timeout=10).status_code == 200
    except Exception: return False

_hub.has_file = _safe_has_file
_mutils.has_file = _safe_has_file

# Entry 7 — FinBERT fine-tuned ---------------------------------------------
if ENABLE_FINBERT:
    try:
        import torch
        from transformers import (BertTokenizerFast, AutoModelForSequenceClassification,
                                  TrainingArguments as HFArgs, Trainer as HFTrainer)
        from datasets import Dataset
        HAS_HF = True
    except Exception as e:
        HAS_HF = False
        print("transformers not available:", e)

    if HAS_HF:
        FINBERT_BASE = "yiyanghkust/finbert-tone"
        DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

        def fit_finbert(texts, labels, output_dir: Path):
            tok = BertTokenizerFast.from_pretrained(FINBERT_BASE)
            model = AutoModelForSequenceClassification.from_pretrained(
                FINBERT_BASE, num_labels=2, ignore_mismatched_sizes=True,
            ).to(DEVICE)

            def tok_fn(batch):
                return tok(batch["text"], truncation=True, padding="max_length", max_length=96)

            ds = (Dataset.from_dict({"text": list(texts), "label": list(labels)})
                          .map(tok_fn, batched=True))
            ds = ds.remove_columns(["text"])
            ds.set_format("torch")

            args = HFArgs(
                output_dir=str(output_dir), num_train_epochs=3,
                per_device_train_batch_size=16, learning_rate=2e-5,
                logging_steps=50, save_strategy="no", report_to="none",
                seed=SEED, dataloader_num_workers=0,
            )
            HFTrainer(model=model, args=args, train_dataset=ds).train()
            return model, tok

        @torch.no_grad()
        def finbert_proba(model, tok, texts) -> np.ndarray:
            model.eval()
            out = []
            BS = 32
            for i in range(0, len(texts), BS):
                batch = list(texts[i:i + BS])
                enc = tok(batch, truncation=True, padding=True,
                          max_length=96, return_tensors="pt")
                # explicitly cast to long and move to device
                enc = {k: v.long().to(DEVICE) if v.dtype != torch.long
                       else v.to(DEVICE) for k, v in enc.items()}
                logits = model(**{k: v for k, v in enc.items()
                                  if k in ("input_ids","attention_mask","token_type_ids")}).logits
                p = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
                out.extend(p.tolist())
            return np.asarray(out)

        oof = np.zeros(len(y_pool))
        t0 = time.perf_counter()
        for fold, (tr, te) in enumerate(cv_indices()):
            m, tk = fit_finbert([texts_pool[i] for i in tr], y_pool[tr],
                                CACHE_DIR / f"finbert_fold{fold}")
            oof[te] = finbert_proba(m, tk, [texts_pool[i] for i in te])
            del m, tk
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        t_train = time.perf_counter() - t0

        m_final, tk_final = fit_finbert(texts_pool, y_pool, CACHE_DIR / "finbert_full")
        sps = time_inference(lambda X: finbert_proba(m_final, tk_final, X), texts_test)
        test_p = finbert_proba(m_final, tk_final, texts_test)

        ENTRIES["finbert"] = Entry(
            name="finbert", family="Transformer",
            description="FinBERT (yiyanghkust/finbert-tone) fine-tuned, 3 epochs LR 2e-5",
            train_seconds=t_train, sentences_per_second=sps,
            oof_proba=oof, test_proba=test_p,
        )
        print("finbert OK")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at yiyanghkust/finbert-tone and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 960/960 [00:00<00:00, 7408.78 examples/s]


Step,Training Loss
50,0.431000
100,0.210000
150,0.110000


RuntimeError: Placeholder storage has not been allocated on MPS device!

In [27]:

# import subprocess
# import sys

# print("Attempting to fix the huggingface_hub / requests version mismatch...")
# print("(This installs into the kernel's environment; you may need to restart")
# print(" the kernel afterwards.)\n")

# # Upgrade huggingface_hub to a recent version that's compatible with both
# # transformers 4.x and the new requests/httpx interface
# cmds = [
#     [sys.executable, "-m", "pip", "install", "-q", "-U",
#      "huggingface_hub>=0.24.0", "requests>=2.32.0"],
# ]
# for cmd in cmds:
#     print(f"  $ {' '.join(cmd[3:])}")
#     r = subprocess.run(cmd, capture_output=True, text=True)
#     if r.returncode != 0:
#         print(f"  -> FAILED:\n{r.stderr[-500:]}")
#         break
#     else:
#         print(f"  -> OK")

# print()
# print("Now restart the kernel (Kernel -> Restart) and re-run from cell 1.")
# print("If FinBERT still errors with the same message, try the alternative")
# print("model 'ProsusAI/finbert' instead — change FINBERT_BASE in cell 32:")
# print()
# print('    FINBERT_BASE = "ProsusAI/finbert"')
# print()
# print("This is a different but comparable financial BERT, downloaded fresh")
# print("with the upgraded huggingface_hub it should work.")


In [28]:
# Entry 8 — Prototype-cosine classifier (creative, zero-training) ----------
# For each class, we keep K=24 anchor sentences with the highest mean within-class
# similarity (the "prototypes"). At inference we score by max cosine similarity to
# either set, then turn the gap into a probability via a sigmoid.
def select_prototypes(emb: np.ndarray, k: int = 24) -> np.ndarray:
    """Greedy k-medoids-style: pick the k sentences with the highest sum of
    cosine similarity to all other sentences in the class."""
    sims = emb @ emb.T                           # cosine (already normalized)
    centrality = sims.sum(axis=1)
    return np.argsort(-centrality)[:k]


t0 = time.perf_counter()
proto_idx_pos = select_prototypes(X_pool_emb[y_pool == 1])
proto_idx_neg = select_prototypes(X_pool_emb[y_pool == 0])
proto_pos = X_pool_emb[y_pool == 1][proto_idx_pos]
proto_neg = X_pool_emb[y_pool == 0][proto_idx_neg]
t_train = time.perf_counter() - t0


def proto_score(X: np.ndarray) -> np.ndarray:
    s_pos = (X @ proto_pos.T).max(axis=1)
    s_neg = (X @ proto_neg.T).max(axis=1)
    z = (s_pos - s_neg) * 8.0                    # temperature
    return 1.0 / (1.0 + np.exp(-z))


def proto_oof(X_full, y, splits):
    oof = np.zeros(len(y))
    for tr, te in splits:
        Xtr, ytr = X_full[tr], y[tr]
        idx_pos = select_prototypes(Xtr[ytr == 1])
        idx_neg = select_prototypes(Xtr[ytr == 0])
        Pp, Pn = Xtr[ytr == 1][idx_pos], Xtr[ytr == 0][idx_neg]
        s_pos = (X_full[te] @ Pp.T).max(axis=1)
        s_neg = (X_full[te] @ Pn.T).max(axis=1)
        oof[te] = 1.0 / (1.0 + np.exp(-(s_pos - s_neg) * 8.0))
    return oof


oof = proto_oof(X_pool_emb, y_pool, cv_indices())
sps = time_inference(proto_score, X_test_emb)
test_p = proto_score(X_test_emb)

ENTRIES["prototype_cosine"] = Entry(
    name="prototype_cosine", family="Anchor (creative)",
    description="K-medoid prototypes per class; max-cosine gap → sigmoid",
    train_seconds=t_train, sentences_per_second=sps,
    oof_proba=oof, test_proba=test_p,
    notes="No gradient descent. Interpretable: which prototype was closest?",
)
print("prototype_cosine OK")


prototype_cosine OK


In [29]:
# Entry 9 — Two-stage triage (creative) -----------------------------------
# Stage 1: strong rule signals decide obvious cases (very high precision flags only).
# Stage 2: the embedding LogReg classifies the rest.
#
# Goal: push substantive recall up by NEVER calling a sentence boilerplate that
# contains a hard substantive cue ($, %, bps, guidance verb), regardless of what
# the model thinks.
HARD_SUB_KEYS = ["has_dollar", "has_percent", "has_bps", "has_million_billion",
                 "has_year", "has_quarter", "guidance_word", "margin_word"]
HARD_BOI_KEYS = ["starts_with_operator", "mute_lines", "queue_phrase",
                 "recording_phrase", "safe_harbor", "forward_looking",
                 "name_intro", "analyst_firm"]


def two_stage_score(rx_df: pd.DataFrame, emb: np.ndarray, base_model) -> np.ndarray:
    base_p = base_model.predict_proba(emb)[:, 1]
    sub_hits = rx_df[HARD_SUB_KEYS].sum(axis=1).values
    boi_hits = rx_df[HARD_BOI_KEYS].sum(axis=1).values
    out = base_p.copy()
    out = np.where(sub_hits >= 1, np.maximum(out, 0.92), out)   # force ≥ 0.92 if hard sub cue
    out = np.where((sub_hits == 0) & (boi_hits >= 2),
                   np.minimum(out, 0.10), out)                  # force ≤ 0.10 if 2+ boi cues and no sub cue
    return out


t0 = time.perf_counter()
oof = np.zeros(len(y_pool))
for tr, te in cv_indices():
    base = make_logreg_emb().fit(X_pool_emb[tr], y_pool[tr])
    oof[te] = two_stage_score(X_pool_rx.iloc[te], X_pool_emb[te], base)
t_train = time.perf_counter() - t0

base_full = make_logreg_emb().fit(X_pool_emb, y_pool)
def stage_predict(emb): return two_stage_score(X_test_rx, emb, base_full)
sps = time_inference(stage_predict, X_test_emb)
test_p = two_stage_score(X_test_rx, X_test_emb, base_full)

ENTRIES["two_stage"] = Entry(
    name="two_stage", family="Hybrid (creative)",
    description="Rule overrides on hard cues + LogReg(embeddings) for the rest",
    train_seconds=t_train, sentences_per_second=sps,
    oof_proba=oof, test_proba=test_p,
    notes="Designed for the recall floor — hard $%/bps cues never get downgraded.",
)
print("two_stage OK")


two_stage OK


In [30]:
# Entry 10 — Soft-label distillation LogReg (creative) ---------------------
# Use the mean of the three judges' P(substantive) as a SOFT TARGET. Train a
# regularized regression on embeddings; threshold its output as a probability.
# We weight each sample by judge agreement so noisy labels contribute less.
from sklearn.linear_model import Ridge

# Build mean-judge probability per sentence in the pool
votes_pool = votes_df[votes_df["sentence_id"].isin(df_pool["sentence_id"])].copy()
votes_pool["p_sub"] = np.where(votes_pool["label"] == "substantive",
                               votes_pool["confidence"],
                               1.0 - votes_pool["confidence"])
mean_p = votes_pool.groupby("sentence_id")["p_sub"].mean()
agree  = votes_pool.groupby("sentence_id")["label"].apply(
            lambda s: int(s.value_counts().iloc[0] == len(s)))   # 1 if unanimous

soft_target = df_pool["sentence_id"].map(mean_p).fillna(df_pool["y"]).values
sample_w    = 0.6 + 0.4 * df_pool["sentence_id"].map(agree).fillna(0.5).values  # ∈ [0.6, 1.0]


def make_distill():
    return Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", Ridge(alpha=1.0, random_state=SEED)),
    ])


def to_proba(z):  # squash ridge output
    return 1.0 / (1.0 + np.exp(-(z * 4.0 - 2.0)))


oof = np.zeros(len(y_pool))
t0 = time.perf_counter()
for tr, te in cv_indices():
    m = make_distill()
    m.fit(X_pool_emb[tr], soft_target[tr], clf__sample_weight=sample_w[tr])
    oof[te] = to_proba(m.predict(X_pool_emb[te]))
t_train = time.perf_counter() - t0

final = make_distill().fit(X_pool_emb, soft_target, clf__sample_weight=sample_w)
def distill_predict(X): return to_proba(final.predict(X))
sps = time_inference(distill_predict, X_test_emb)
test_p = distill_predict(X_test_emb)

ENTRIES["distill_softlabel"] = Entry(
    name="distill_softlabel", family="Distillation (creative)",
    description="Ridge on embeddings, fit to mean-judge probability with agreement weights",
    train_seconds=t_train, sentences_per_second=sps,
    oof_proba=oof, test_proba=test_p,
    notes="Inference uses only embeddings — no LLM calls at deploy time.",
)
print("distill_softlabel OK")


distill_softlabel OK


In [31]:
# Entry 11 — Mean-probability ensemble of top-5 non-transformer ------------
def macro_f1_oof(entry: Entry, threshold: float = 0.5) -> float:
    return f1_score(y_pool, (entry.oof_proba >= threshold).astype(int), average="macro")


# Rank current entries by OOF macro-F1 at default threshold; pick top 5 non-transformer.
ranked = sorted([e for e in ENTRIES.values() if e.family != "Transformer"],
                key=macro_f1_oof, reverse=True)[:5]
print("Mean ensemble members:", [e.name for e in ranked])

mean_oof  = np.mean([e.oof_proba  for e in ranked], axis=0)
mean_test = np.mean([e.test_proba for e in ranked], axis=0)

ENTRIES["mean_ensemble"] = Entry(
    name="mean_ensemble", family="Ensemble",
    description=f"Mean-probability of top-5: {[e.name for e in ranked]}",
    train_seconds=sum(e.train_seconds for e in ranked),
    sentences_per_second=min(e.sentences_per_second for e in ranked),
    oof_proba=mean_oof, test_proba=mean_test,
    notes="Cheap to combine; usually 1–3 F1 points over the best single member.",
)
print("mean_ensemble OK")


Mean ensemble members: ['hgb_combined', 'setfit', 'logreg_embed', 'two_stage', 'svm_charngram']
mean_ensemble OK


In [32]:
# Entry 12 — Stacked meta-LogReg on OOF probabilities (creative) -----------
# Train a meta-classifier on the OOF probabilities of the same top-5 members.
# Because OOF probs are unbiased (each fold's prediction came from a model that
# didn't see the held-out points), this is honest stacking.
from sklearn.linear_model import LogisticRegression as LR

X_meta_pool = np.column_stack([e.oof_proba  for e in ranked])
X_meta_test = np.column_stack([e.test_proba for e in ranked])

t0 = time.perf_counter()
meta_oof = np.zeros(len(y_pool))
for tr, te in cv_indices():
    m = LR(max_iter=500, C=1.0).fit(X_meta_pool[tr], y_pool[tr])
    meta_oof[te] = m.predict_proba(X_meta_pool[te])[:, 1]
meta = LR(max_iter=500, C=1.0).fit(X_meta_pool, y_pool)
test_p = meta.predict_proba(X_meta_test)[:, 1]
t_train = time.perf_counter() - t0

ENTRIES["stacked_meta"] = Entry(
    name="stacked_meta", family="Ensemble",
    description=f"LogReg stacked on OOF probs of top-5: {[e.name for e in ranked]}",
    train_seconds=t_train,
    sentences_per_second=min(e.sentences_per_second for e in ranked),
    oof_proba=meta_oof, test_proba=test_p,
    notes="Learns weights instead of averaging. More flexible than mean ensemble.",
)
print("stacked_meta OK")
print("\\nAll entries:", list(ENTRIES))


stacked_meta OK
\nAll entries: ['rules_only', 'logreg_embed', 'svm_charngram', 'hgb_combined', 'fasttext', 'setfit', 'prototype_cosine', 'two_stage', 'distill_softlabel', 'mean_ensemble', 'stacked_meta']


## 7 · Recall-constrained threshold tuning

For each entry: sweep the decision threshold over `[0.05, 0.95]`, find the **largest** threshold satisfying substantive recall ≥ 0.96 on pooled OOF predictions, then among all feasible thresholds pick the one with maximum macro-F1.

Reports per-fold std of the optimal threshold (the handout asks for it). Entries that cannot meet the floor are flagged `INFEASIBLE` — we do **not** silently relax the constraint.


In [33]:
def per_fold_best_threshold(probs: np.ndarray, y: np.ndarray,
                            splits, recall_floor: float) -> tuple[float, float]:
    """Return (mean, std) of per-fold best thresholds that meet the floor.
    Returns (nan, nan) if no fold can meet the floor."""
    fold_thresholds = []
    grid = np.linspace(0.02, 0.98, 97)
    for _, te in splits:
        ys = y[te]; ps = probs[te]
        feasible = []
        for t in grid:
            yhat = (ps >= t).astype(int)
            if recall_score(ys, yhat, pos_label=1, zero_division=0) >= recall_floor:
                feasible.append((t, f1_score(ys, yhat, average="macro", zero_division=0)))
        if feasible:
            fold_thresholds.append(max(feasible, key=lambda x: x[1])[0])
    if not fold_thresholds:
        return float("nan"), float("nan")
    return float(np.mean(fold_thresholds)), float(np.std(fold_thresholds))


def tune_threshold(probs: np.ndarray, y: np.ndarray,
                   recall_floor: float = RECALL_FLOOR):
    grid = np.linspace(0.02, 0.98, 97)
    feasible = []
    for t in grid:
        yhat = (probs >= t).astype(int)
        if recall_score(y, yhat, pos_label=1, zero_division=0) >= recall_floor:
            feasible.append((t, f1_score(y, yhat, average="macro", zero_division=0)))
    if not feasible:
        return None, None, "INFEASIBLE"
    best_t, best_f1 = max(feasible, key=lambda x: x[1])
    return best_t, best_f1, "OK"


splits = cv_indices()
threshold_results = {}
for name, e in ENTRIES.items():
    best_t, best_f1, status = tune_threshold(e.oof_proba, y_pool, RECALL_FLOOR)
    fold_mean, fold_std = per_fold_best_threshold(e.oof_proba, y_pool, splits, RECALL_FLOOR)
    threshold_results[name] = {
        "threshold":      best_t,
        "oof_macroF1":    best_f1,
        "status":         status,
        "fold_mean":      fold_mean,
        "fold_std":       fold_std,
    }

pd.DataFrame(threshold_results).T.round(4)


,threshold,oof_macroF1,status,fold_mean,fold_std
rules_only,0.27,0.38519,OK,0.27,0.0
logreg_embed,None,None,INFEASIBLE,0.03,0.0
svm_charngram,0.24,0.680227,OK,0.26,0.055136
hgb_combined,None,None,INFEASIBLE,0.04,0.0
fasttext,0.22,0.63625,OK,0.27,0.172511
setfit,0.27,0.787765,OK,0.268,0.0801
prototype_cosine,0.3,0.56626,OK,0.312,0.022271
two_stage,None,None,INFEASIBLE,0.065,0.045
distill_softlabel,0.16,0.626604,OK,0.17,0.04858
mean_ensemble,0.2,0.773356,OK,0.22,0.081486


## 8 · Held-out test evaluation and leaderboard

Apply each entry's tuned threshold to the **frozen** test set. Report test accuracy, macro-F1, per-class F1, training time, and approximate inference throughput. Sorted by macro-F1 descending, with infeasible entries listed at the bottom.


In [ ]:
rows = []
for name, e in ENTRIES.items():
    tr = threshold_results[name]
    if tr["status"] == "INFEASIBLE":
        rows.append({
            "name": name, "family": e.family, "status": "INFEASIBLE",
            "threshold": np.nan, "test_acc": np.nan, "test_macroF1": np.nan,
            "boi_F1": np.nan, "sub_F1": np.nan, "sub_recall": np.nan, "sub_prec": np.nan,
            "train_sec": e.train_seconds, "sent_per_sec": e.sentences_per_second,
            "fold_threshold_std": tr["fold_std"], "description": e.description,
        })
        continue
    test_proba = np.asarray(e.test_proba).reshape(-1)
    yhat = (test_proba >= tr["threshold"]).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_test, yhat, labels=[0, 1], zero_division=0)
    rows.append({
        "name": name, "family": e.family, "status": "OK",
        "threshold": tr["threshold"],
        "test_acc": (yhat == y_test).mean(),
        "test_macroF1": f1_score(y_test, yhat, average="macro", zero_division=0),
        "boi_F1": f[0], "sub_F1": f[1], "sub_recall": r[1], "sub_prec": p[1],
        "train_sec": e.train_seconds, "sent_per_sec": e.sentences_per_second,
        "fold_threshold_std": tr["fold_std"], "description": e.description,
    })

LEADERBOARD = pd.DataFrame(rows)
LEADERBOARD["status_order"] = np.where(LEADERBOARD["status"] == "OK", 0, 1)
LEADERBOARD = (LEADERBOARD
               .sort_values(["status_order", "test_macroF1"], ascending=[True, False])
               .drop(columns=["status_order"])
               .reset_index(drop=True))
LEADERBOARD.to_csv(REPORTS_DIR / "leaderboard.csv", index=False)

display_cols = ["name", "family", "status", "threshold", "test_macroF1",
                "boi_F1", "sub_F1", "sub_recall", "sub_prec",
                "train_sec", "sent_per_sec", "fold_threshold_std"]
LEADERBOARD[display_cols].round(4)


,name,family,status,threshold,test_macroF1,boi_F1,sub_F1,sub_recall,sub_prec,train_sec,sent_per_sec,fold_threshold_std
0,mean_ensemble,Ensemble,OK,0.20,0.7758,0.7265,0.8251,0.9618,0.7225,383.1050,101.2673,0.0815
1,setfit,Contrastive,OK,0.27,0.7603,0.7043,0.8162,0.9618,0.7089,350.7790,101.2673,0.0801
2,stacked_meta,Ensemble,OK,0.14,0.7532,0.6957,0.8108,0.9554,0.7042,0.0093,101.2673,0.0932
3,svm_charngram,Linear (lexical),OK,0.24,0.6703,0.5659,0.7747,0.9745,0.6429,0.9420,8578.0359,0.0551
4,fasttext,N-gram,OK,0.22,0.6080,0.4747,0.7413,0.9490,0.6082,2.9701,79834.1367,0.1826
5,distill_softlabel,Distillation (creative),OK,0.16,0.5689,0.4022,0.7356,0.9745,0.5907,0.0741,825025.7731,0.0486
6,prototype_cosine,Anchor (creative),OK,0.30,0.5454,0.3696,0.7212,0.9554,0.5792,0.0088,592203.0492,0.0223
7,rules_only,Rules,OK,0.27,0.4460,0.1875,0.7045,0.9873,0.5477,0.0019,367497.0915,0.0000
8,logreg_embed,Linear,INFEASIBLE,NaN,NaN,NaN,NaN,NaN,NaN,0.1832,428291.2059,0.0000
9,hgb_combined,Tree ensemble,INFEASIBLE,NaN,NaN,NaN,NaN,NaN,NaN,31.0190,25670.7379,0.0000


In [ ]:
# ---- Confusion matrix and full classification report for the winner ------
feasible = LEADERBOARD[LEADERBOARD["status"] == "OK"]
assert len(feasible) > 0, "No classifier met the recall floor — collect more gold or pick a stronger model."
winner = feasible.iloc[0]
print(f"Winner: {winner['name']}  (test macro-F1 = {winner['test_macroF1']:.4f}, "
      f"threshold = {winner['threshold']:.3f})\n")

w_entry = ENTRIES[winner["name"]]
yhat = (w_entry.test_proba >= winner["threshold"]).astype(int)
print(classification_report(y_test, yhat,
                            target_names=["boilerplate", "substantive"], digits=4))
print("Confusion matrix (rows = truth, cols = pred):")
print(pd.DataFrame(confusion_matrix(y_test, yhat),
                   index=["true_boilerplate", "true_substantive"],
                   columns=["pred_boilerplate", "pred_substantive"]))


Winner: mean_ensemble  (test macro-F1 = 0.7758, threshold = 0.200)

              precision    recall  f1-score   support

 boilerplate     0.9341    0.5944    0.7265       143
 substantive     0.7225    0.9618    0.8251       157

    accuracy                         0.7867       300
   macro avg     0.8283    0.7781    0.7758       300
weighted avg     0.8233    0.7867    0.7781       300

Confusion matrix (rows = truth, cols = pred):
                  pred_boilerplate  pred_substantive
true_boilerplate                85                58
true_substantive                 6               151


## 9 · Save the winning bundle

Persist the winning model along with the threshold, the regex feature pipeline (if used), and a tiny inference helper. The GUI loads `models/bp_best.joblib` at startup.


In [ ]:
# ---- Bundle saver --------------------------------------------------------
BUNDLE_PATH = MODELS_DIR / "bp_best.joblib"


def fit_member_artifact(name: str) -> dict:
    """Fit and store the pieces needed to reproduce a member's test-time probability."""
    if name == "logreg_embed":
        return {"estimator": make_logreg_emb().fit(X_pool_emb, y_pool)}
    if name == "svm_charngram":
        return {"estimator": make_svm_tfidf().fit(texts_pool, y_pool)}
    if name == "hgb_combined":
        return {"estimator": make_hgb().fit(X_pool_full, y_pool)}
    if name == "distill_softlabel":
        return {"estimator": make_distill().fit(X_pool_emb, soft_target,
                                                  clf__sample_weight=sample_w)}
    if name == "fasttext":
        model_path = MODELS_DIR / "fasttext_member.bin"
        ft_model = train_ft(texts_pool, y_pool, CACHE_DIR / "ft_train_bundle.txt")
        ft_model.save_model(str(model_path))
        return {"model_path": str(model_path)}
    if name == "setfit":
        if "SETFIT_MODEL_DIR" not in globals():
            raise RuntimeError("SetFit model directory is unavailable; rerun the SetFit cell before bundling.")
        return {"model_dir": str(SETFIT_MODEL_DIR)}
    if name == "two_stage":
        return {"base_estimator": make_logreg_emb().fit(X_pool_emb, y_pool)}
    if name == "prototype_cosine":
        return {"prototypes": {"pos": proto_pos, "neg": proto_neg}}
    if name == "rules_only":
        return {}
    raise NotImplementedError(f"Cannot bundle member {name!r}")


bundle = {
    "winner_name":    winner["name"],
    "threshold":      float(winner["threshold"]),
    "embed_model":    EMBED_MODEL_NAME,
    "regex_features": REGEX_FEATURE_NAMES,
    "feature_patterns": FEATURE_PATTERNS,
    "test_metrics": {
        "macro_f1":   float(winner["test_macroF1"]),
        "sub_recall": float(winner["sub_recall"]),
        "sub_prec":   float(winner["sub_prec"]),
        "boi_f1":     float(winner["boi_F1"]),
        "sub_f1":     float(winner["sub_F1"]),
    },
    "leaderboard":  LEADERBOARD,
    "trained_at":   pd.Timestamp.utcnow().isoformat(),
}

name = winner["name"]
if name in {"mean_ensemble", "stacked_meta"}:
    members = [e.name for e in ranked]
    bundle["ensemble_members"] = members
    bundle["member_artifacts"] = {member: fit_member_artifact(member) for member in members}
    if name == "stacked_meta":
        bundle["meta_estimator"] = LR(max_iter=500, C=1.0).fit(X_meta_pool, y_pool)
elif name == "logreg_embed":
    bundle.update(fit_member_artifact("logreg_embed"))
elif name == "svm_charngram":
    bundle.update(fit_member_artifact("svm_charngram"))
elif name == "hgb_combined":
    bundle.update(fit_member_artifact("hgb_combined"))
elif name == "distill_softlabel":
    bundle.update(fit_member_artifact("distill_softlabel"))
elif name == "fasttext":
    bundle.update(fit_member_artifact("fasttext"))
elif name == "setfit":
    bundle.update(fit_member_artifact("setfit"))
elif name == "prototype_cosine":
    bundle.update(fit_member_artifact("prototype_cosine"))
elif name == "two_stage":
    bundle.update(fit_member_artifact("two_stage"))
elif name == "rules_only":
    pass
else:
    raise NotImplementedError(f"Bundle saving for {name!r} is not wired up.")

joblib.dump(bundle, BUNDLE_PATH)
print(f"Saved bundle -> {BUNDLE_PATH}")
print(f"  winner={bundle['winner_name']}  threshold={bundle['threshold']:.3f}  "
      f"macro-F1={bundle['test_metrics']['macro_f1']:.4f}")


Saved bundle -> /Users/chaithanyapakala/Documents/NLP/Pakala_Chaithanya_NLP_HW2/models/bp_best.joblib
  winner=mean_ensemble  threshold=0.200  macro-F1=0.7758


## 10 · Sanity check on a fresh transcript

Pick a transcript the model never saw (anything not in the gold pool's `transcript` column counts) and tag it inline. This is what the Streamlit GUI will do once we ship it in your remaining-steps pass.


In [ ]:
def predict_label(sentences: list[str]) -> tuple[np.ndarray, np.ndarray]:
    """Returns (probs_substantive, predicted_labels) using the saved bundle."""
    from sentence_transformers import SentenceTransformer
    bundle = joblib.load(BUNDLE_PATH)
    embedder = SentenceTransformer(bundle["embed_model"])
    emb = embedder.encode(sentences, batch_size=64, normalize_embeddings=True,
                          show_progress_bar=False, convert_to_numpy=True)
    rx = pd.DataFrame([regex_features(s) for s in sentences], columns=REGEX_FEATURE_NAMES)

    def score_member(member: str) -> np.ndarray:
        artifacts = bundle.get("member_artifacts", {}).get(member, bundle)
        if member == "logreg_embed":
            return artifacts["estimator"].predict_proba(emb)[:, 1]
        if member == "svm_charngram":
            return artifacts["estimator"].predict_proba(sentences)[:, 1]
        if member == "hgb_combined":
            full = np.hstack([emb, rx.values])
            return artifacts["estimator"].predict_proba(full)[:, 1]
        if member == "two_stage":
            return two_stage_score(rx, emb, artifacts["base_estimator"])
        if member == "prototype_cosine":
            protos = artifacts["prototypes"]
            s_pos = (emb @ protos["pos"].T).max(axis=1)
            s_neg = (emb @ protos["neg"].T).max(axis=1)
            return 1.0 / (1.0 + np.exp(-(s_pos - s_neg) * 8.0))
        if member == "rules_only":
            return rules_score(rx)
        if member == "distill_softlabel":
            return to_proba(artifacts["estimator"].predict(emb))
        if member == "fasttext":
            import fasttext
            ft_model = fasttext.load_model(artifacts["model_path"])
            return ft_proba(ft_model, sentences)
        if member == "setfit":
            from setfit import SetFitModel
            sf_model = SetFitModel.from_pretrained(artifacts["model_dir"])
            p = np.asarray(sf_model.predict_proba(list(sentences)))
            return p[:, 1] if p.ndim == 2 and p.shape[1] == 2 else p.reshape(-1)
        raise NotImplementedError(f"Inference for member {member!r} is not wired up.")

    name = bundle["winner_name"]
    if name in {"mean_ensemble", "stacked_meta"}:
        member_probs = np.column_stack([score_member(m) for m in bundle["ensemble_members"]])
        if name == "mean_ensemble":
            probs = member_probs.mean(axis=1)
        else:
            probs = bundle["meta_estimator"].predict_proba(member_probs)[:, 1]
    else:
        probs = score_member(name)

    yhat = (probs >= bundle["threshold"]).astype(int)
    return probs, yhat


# Pick the first transcript not represented in our gold set and tag it.
unused_transcripts = sorted(set(sentences_df["transcript"]) - set(gold["transcript"]))
sample_transcript = unused_transcripts[0] if unused_transcripts else sentences_df["transcript"].iloc[0]
sample = sentences_df[sentences_df["transcript"] == sample_transcript].head(20)

probs, yhat = predict_label(sample["sentence"].tolist())
preview = sample.assign(p_substantive=probs.round(3),
                        prediction=np.where(yhat == 1, "substantive", "boilerplate"))
preview[["sentence", "p_substantive", "prediction"]].head(15)


,sentence,p_substantive,prediction
0,"﻿Advanced Micro Devices, Inc., Q1 2024 Earning...",0.649,substantive
1,Presentation Operator Message Operator Greetin...,0.412,substantive
2,"[Operator Instructions] As a reminder, this co...",0.143,boilerplate
3,"It is now my pleasure to introduce your host, ...",0.067,boilerplate
4,Presenter Speech Executives - Former Vice Pres...,0.476,substantive
5,"By now, you should have had the opportunity to...",0.133,boilerplate
6,If you have not had the chance to review these...,0.226,substantive
7,We will refer primarily to non-GAAP financial ...,0.326,substantive
8,"Participants on today's call are Dr. Lisa Su, ...",0.071,boilerplate
9,This is a live call and will be replayed via w...,0.203,substantive


## 11 · Next steps for the remaining-steps pass

When you're ready, ping me to do:

1. **`app.py`** — Streamlit GUI that loads `models/bp_best.joblib`, accepts a transcript, renders sentences inline with boilerplate highlighted in red, and shows the count/percentage stats panel.
2. **Write-up PDF** — 5–10 pages following the handout's outline (intro, gold methodology, feature engineering, leaderboard, threshold-tuning narrative, error analysis, GUI screenshot, reproducibility commands).
3. **Final zip** packaged the way Howard wants it.

Things you can do now to make those passes easy:
- Confirm the leaderboard above looks reasonable to you.
- Skim `reports/disagreement_audit.csv` and override any obvious labeling mistakes (then re-run the freeze cell).
- Run the sanity check on one or two transcripts you haven't seen yet to make sure the inline tags look right.
